# Session S1 — Setup + CPU Baselines

**DS-PatchMamba | WORKFLOW.md Session S1**  
**Session target: ~6–8 GPU-hours (CPU baselines use no GPU)**

### What this session does
1. Installs packages and downloads TSB-AD-M
2. Loads datasets and verifies shapes and label alignment
3. Runs 5 CPU baselines: Random, PCA, IForest, OLS-RRR, T²-VAR
4. Validates the evaluation harness against known VUS-PR ranges

### Before running — two manual steps on Kaggle
**Step A — Add legacy datasets as Kaggle inputs:**
Search for and add these three public Kaggle datasets to your notebook's input:
- `Server Machine Dataset (SMD)` — mounted at `/kaggle/input/smd/`
- `PSM anomaly detection` — mounted at `/kaggle/input/psm/`
- `SWaT dataset` — mounted at `/kaggle/input/swat/`

**Step B — Code from GitHub (automatic):**
Cell 1 clones `https://github.com/bikesh2002/ds-patchmamba` automatically.
Make sure the repo is **public** so Kaggle can clone without authentication.

### Session-death protection (12-hour cap)
- Results appended to `results/main_results.csv` row-by-row after each series
- Scores saved as `.npy` files immediately after inference
- Skip-logic reads the CSV before each run — already-done runs are skipped
- Click **Save Version** before the session ends to persist `/kaggle/working/`

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Environment setup
# Expected time: ~10–15 minutes
# ══════════════════════════════════════════════════════════════════════════════

import os, sys, subprocess, shutil, time

# ── Restore outputs from a previous saved session ─────────────────────────
# When you resume from a Kaggle saved version, your previous results/
# and checkpoints/ folders are in /kaggle/input/ds-patchmamba-runN/.
# We copy them into the working directory so the skip-logic in Cell 3
# can see what has already been completed.
for prev_run in [
    "/kaggle/input/ds-patchmamba-run2",
    "/kaggle/input/ds-patchmamba-run1",
]:
    if os.path.exists(prev_run):
        shutil.copytree(prev_run, "/kaggle/working/", dirs_exist_ok=True)
        print(f"Restored previous session outputs from: {prev_run}")
        break

# ── Clone src/ from GitHub ─────────────────────────────────────────────────
# Replace YOUR_USERNAME with your actual GitHub username.
REPO_URL = "https://github.com/bikesh2002/ds-patchmamba.git"

# Always fresh clone — avoids 'divergent branches' git pull errors on Kaggle.
REPO_DIR = "/kaggle/working/repo"
SRC_LINK = "/kaggle/working/src"

if os.path.islink(SRC_LINK):
    os.unlink(SRC_LINK)
elif os.path.exists(SRC_LINK):
    shutil.rmtree(SRC_LINK)
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
    check=True,
)
os.symlink(f"{REPO_DIR}/src", SRC_LINK)
print("Cloned latest src/ from GitHub.")

# Sync critical files directly from GitHub main (bypasses git pull / divergent branch issues)
import urllib.request
for rel in ["src/data/loader.py", "src/data/preprocessing.py", "src/evaluation/metrics.py"]:
    url  = f"https://raw.githubusercontent.com/bikesh2002/ds-patchmamba/main/{rel}"
    dest = f"/kaggle/working/{rel}"
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    try:
        urllib.request.urlretrieve(url, dest)
        print(f"  Synced {rel}")
    except Exception as e:
        print(f"  [WARN] Could not sync {rel}: {e}")

if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

# ── Create output directories ──────────────────────────────────────────────
for d in ["results/scores", "checkpoints", "data"]:
    os.makedirs(d, exist_ok=True)

# ── Install Python packages ────────────────────────────────────────────────
# IMPORTANT (Kaggle): Do NOT install TSB-AD here.
# TSB-AD pulls numpy <2.0, which breaks Kaggle's pre-built pandas and causes:
#   ValueError: numpy.dtype size changed, may indicate binary incompatibility
# Our metrics.py has a built-in VUS-PR fallback — Session 1 works without TSB-AD.
print("Installing packages...")
pkgs = ["statsmodels", "scikit-posthocs", "PyWavelets"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=False)
print("Packages installed (VUS-PR will use built-in fallback, not TSB-AD package).")

# ── Download TSB-AD-M series files ───────────────────────────────────────────
# The zip contains only the time-series CSV files (~2 GB extracted).
# Eval/Tuning split lists are NOT in the zip — they live on the TSB-AD GitHub repo.
tsb_eva    = "data/TSB-AD-M-Eva.csv"
tsb_tuning = "data/TSB-AD-M-Tuning.csv"
FILE_LIST_BASE = (
    "https://raw.githubusercontent.com/TheDatumOrg/TSB-AD/main/Datasets/File_List"
)

# Step A: series CSVs from zip (skip if already extracted)
_series_present = (
    os.path.isdir("data/TSB-AD-M")
    or any(f.endswith(".csv") for f in os.listdir("data")
           if os.path.isfile(os.path.join("data", f)) and f not in ("TSB-AD-M-Eva.csv", "TSB-AD-M-Tuning.csv"))
)
if not _series_present:
    print("Downloading TSB-AD-M series zip...")
    subprocess.run([
        "wget", "-q",
        "https://www.thedatum.org/datasets/TSB-AD-M.zip",
        "-O", "data/TSB-AD-M.zip"
    ], check=True)
    subprocess.run(["unzip", "-o", "-q", "data/TSB-AD-M.zip", "-d", "data/"], check=True)
    print("TSB-AD-M series files extracted.")
else:
    print("TSB-AD-M series files already present.")

# Download official Eval/Tuning index CSVs (always check — not included in zip)
for name, dest in [("TSB-AD-M-Eva", tsb_eva), ("TSB-AD-M-Tuning", tsb_tuning)]:
    if not os.path.exists(dest):
        print(f"Downloading {name}.csv from TSB-AD GitHub...")
        subprocess.run(
            ["wget", "-q", f"{FILE_LIST_BASE}/{name}.csv", "-O", dest],
            check=True,
        )
    print(f"  {name}: {dest} OK")

# ── Verify TSB-AD-M structure ──────────────────────────────────────────────
import pandas as pd
assert os.path.exists(tsb_eva),    f"Missing: {tsb_eva}"
assert os.path.exists(tsb_tuning), f"Missing: {tsb_tuning}"

for f, name in [(tsb_eva, 'Eva'), (tsb_tuning, 'Tuning')]:
    idx = pd.read_csv(f)
    path_col = 'filepath' if 'filepath' in idx.columns else 'file_name'
    assert path_col in idx.columns, (
        f"TSB-AD-M-{name}.csv must contain 'file_name' or 'filepath'. "
        f"Actual columns: {list(idx.columns)}"
    )
    print(f"TSB-AD-M {name} set: {len(idx)} series listed (column: {path_col}).")

# ── Map legacy datasets from their Kaggle input paths ─────────────────────
# Legacy datasets are added as Kaggle inputs (not downloaded).
# These are the expected mount paths — adjust if your dataset names differ.
LEGACY_PATHS = {
    "SMD":  "/kaggle/input/smd-onmiad/ServerMachineDataset",
    "PSM":  "/kaggle/input/pooled-server-metrics-psm/data",
    "SWaT": "/kaggle/input/swat-dataset-secure-water-treatment-system",
}

for ds, path in LEGACY_PATHS.items():
    if os.path.exists(path):
        # Symlink into data/ so loader.py can find it at a predictable location
        dest = f"data/{ds}"
        if not os.path.exists(dest):
            os.symlink(path, dest)
        print(f"{ds}: found at {path}")
    else:
        print(f"[WARNING] {ds} not found at {path}. ")
        print(f"          Add it as a Kaggle input dataset and update LEGACY_PATHS above.")

print("\n[DONE] Setup complete.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Load datasets and verify integrity
# ══════════════════════════════════════════════════════════════════════════════

import importlib
import src.data.loader as _loader
importlib.reload(_loader)   # pick up loader.py changes without kernel restart

from src.data.loader import load_tsb_adm, load_all_legacy
import numpy as np

# ── TSB-AD-M Tuning set ────────────────────────────────────────────────────
# We load ONLY the Tuning set (TSB-AD-M-Tuning.csv) here.
#
# The Eval set (TSB-AD-M-Eva.csv) is NEVER loaded until Session S5–S7
# when we run final experiments. Loading it earlier — even just to look at it —
# would risk inadvertently tuning decisions toward the test distribution
# (data leakage). The 15% Tuning set is large enough (30 series) to validate
# the evaluation harness and tune hyperparameters.
print("Loading TSB-AD-M Tuning set...")
tuning_records = load_tsb_adm("data", split="tuning", normalize=True)
print(f"Loaded {len(tuning_records)} series from TSB-AD-M Tuning set.")

# ── Legacy datasets ────────────────────────────────────────────────────────
print("\nLoading legacy datasets...")
legacy_data = load_all_legacy("data", normalize=True)
for ds_name, records in legacy_data.items():
    print(f"{ds_name}: {len(records)} series")

# ── Integrity checks ───────────────────────────────────────────────────────
# For every loaded series verify:
#   1. test data and labels have the same length (loader enforces this but double-check)
#   2. training data has no NaN or Inf values (corrupted files)
#   3. labels are binary {0, 1} only
#   4. anomaly ratio is reasonable (0% means labels are missing; >50% is suspicious)

all_records = list(tuning_records)
for records in legacy_data.values():
    all_records.extend(records)

print(f"\nRunning integrity checks on {len(all_records)} series...")
issues = []

for r in all_records:
    if len(r.test) != len(r.labels):
        issues.append(f"{r.name}: test/label length mismatch ({len(r.test)} vs {len(r.labels)})")
    if np.any(~np.isfinite(r.train)):
        issues.append(f"{r.name}: NaN or Inf in training data")
    if np.any(~np.isfinite(r.test)):
        issues.append(f"{r.name}: NaN or Inf in test data")
    unique_labels = np.unique(r.labels)
    if not set(unique_labels).issubset({0, 1}):
        issues.append(f"{r.name}: labels contain values outside {{0,1}}: {unique_labels}")
    ratio = r.labels.mean()
    if ratio == 0.0:
        issues.append(f"{r.name}: anomaly ratio is 0% — labels may be missing")
    if ratio > 0.50:
        issues.append(f"{r.name}: anomaly ratio is {ratio:.1%} — suspiciously high")

if issues:
    print("[INTEGRITY ISSUES FOUND — fix before proceeding:]")
    for issue in issues:
        print(f"  ✗ {issue}")
else:
    print("[OK] All integrity checks passed.")

# ── Sample inspection ──────────────────────────────────────────────────────
if tuning_records:
    r = tuning_records[0]
    print(f"\nSample series: {r.name}")
    print(f"  V={r.V} channels | train={r.train.shape} | test={r.test.shape}")
    print(f"  Anomaly ratio: {r.labels.mean():.3f}")
    print(f"  Train mean (should be ~0 after z-score): {r.train.mean():.4f}")
    print(f"  Train std  (should be ~1 after z-score): {r.train.std():.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — CPU baselines on legacy datasets
# Expected time: ~30–60 minutes (no GPU used)
# ══════════════════════════════════════════════════════════════════════════════
#
# We run baselines on SMD, PSM, SWaT (NOT on the Tuning set).
# The Tuning set is only used for hyperparameter selection in Session S4.
#
# Each method produces a 1D anomaly score array aligned to the test series length.
# Scores are saved as .npy files immediately — if the session dies mid-loop,
# we resume from the last saved series (skip-logic via results CSV).

import numpy as np
import time
from src.models.baselines.cpu_baselines import run_all_cpu_baselines
from src.evaluation.metrics import compute_all_metrics
from src.evaluation.harness import save_scores, load_completed_runs, append_result, RESULT_COLUMNS

RESULTS_CSV = "results/main_results.csv"
SCORES_DIR  = "results/scores"

# Load which runs are already done (skip-logic — Rule 4)
completed = load_completed_runs(RESULTS_CSV)
print(f"Already completed: {len(completed)} runs (will be skipped).")

for ds_name, records in legacy_data.items():
    print(f"\n{'='*60}")
    print(f"Dataset: {ds_name} ({len(records)} series)")
    print('='*60)

    for series in records:
        print(f"\n  Series: {series.name} | V={series.V} | "
              f"train={len(series.train):,} | test={len(series.test):,} | "
              f"anomaly_ratio={series.labels.mean():.3f}")

        t0 = time.time()
        scores_dict = run_all_cpu_baselines(series.train, series.test, seed=42)
        print(f"  All baselines finished in {time.time()-t0:.1f}s")

        for method, scores in scores_dict.items():
            run_key = (method, ds_name, series.name, "42", "")
            if run_key in completed:
                print(f"    [SKIP] {method} already done")
                continue

            # Validate length alignment before computing any metric.
            # A mismatch here means the baseline returned the wrong array length.
            if len(scores) != len(series.labels):
                print(
                    f"    [ERROR] {method}: score length {len(scores)} != "
                    f"label length {len(series.labels)}. Skipping this method."
                )
                continue

            # Check for degenerate scores (all identical = no discrimination)
            if np.std(scores) < 1e-10:
                print(f"    [WARN] {method}: all scores are identical ({scores[0]:.4f}). "
                      f"Method may have failed silently.")

            # Rule 2: Save raw scores as .npy immediately
            save_scores(SCORES_DIR, method, ds_name, series.name, 42, scores, series.labels)

            # Compute all 10 metrics
            metrics = compute_all_metrics(scores, series.labels)

            # Rule 3: Append result row-by-row to CSV
            row = {
                "method": method, "dataset": ds_name, "series_name": series.name,
                "seed": "42", "ablation": "",
                "n_params": 0, "flops_per_window": 0,
                "peak_vram_gb": 0, "train_time_s": 0,
                **metrics,
            }
            append_result(RESULTS_CSV, row)
            completed.add(run_key)

            print(f"    {method:10s}: VUS-PR={metrics['vus_pr']:.4f}  "
                  f"DQE={metrics['dqe']:.4f}  AUC-ROC={metrics['auc_roc']:.4f}")

print("\n[DONE] CPU baselines complete.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Validate evaluation harness against known reference ranges
# ══════════════════════════════════════════════════════════════════════════════
#
# Before running expensive GPU experiments, we confirm that the evaluation
# harness is producing sensible numbers by checking PCA against the published
# TSB-AD-M leaderboard (NeurIPS 2024 paper).
#
# Expected ranges (from leaderboard and literature):
#   PCA on SMD:  VUS-PR ~ 0.25–0.38
#   Random:      VUS-PR ~ anomaly ratio of dataset (~0.03–0.05)
#   OLS-RRR:     VUS-PR >= PCA (linear AR should beat PCA reconstruction)
#
# If any check fails, DO NOT proceed to GPU sessions.
# Debug the data loader or metric code first.

import pandas as pd
from src.evaluation.harness import get_summary_table

summary = get_summary_table(RESULTS_CSV, metric="vus_pr")
print("=" * 65)
print("VUS-PR summary (mean across series, seed=42)")
print("=" * 65)
print(summary.to_string(index=False))
print()

# ── Check 1: PCA on SMD ───────────────────────────────────────────────────
pca_smd = summary[(summary["method"] == "PCA") & (summary["dataset"] == "SMD")]
if pca_smd.empty:
    print("[SKIP] PCA/SMD not yet in results — run Cell 3 first.")
else:
    vus = float(pca_smd["vus_pr_mean"].values[0])
    lo, hi = 0.25, 0.40
    status = "OK" if lo <= vus <= hi else "FAIL"
    print(f"[{status}] PCA on SMD: VUS-PR = {vus:.4f}  (expected {lo:.2f}–{hi:.2f})")
    if status == "FAIL":
        print("       DEBUG: check normalisation in loader.py and VUS-PR computation in metrics.py")

# ── Check 2: Random score should be near anomaly ratio ───────────────────
random_rows = summary[summary["method"] == "Random"]
if not random_rows.empty:
    avg_random_vus = float(random_rows["vus_pr_mean"].mean())
    print(f"[INFO] Random score avg VUS-PR = {avg_random_vus:.4f}  (should be low, ~0.03–0.10)")

# ── Check 3: OLS-RRR should be >= PCA on the same datasets ───────────────
for ds in ["SMD", "PSM", "SWaT"]:
    pca_v   = summary[(summary["method"]=="PCA")     & (summary["dataset"]==ds)]["vus_pr_mean"]
    ols_v   = summary[(summary["method"]=="OLS-RRR") & (summary["dataset"]==ds)]["vus_pr_mean"]
    if pca_v.empty or ols_v.empty:
        continue
    pca_val, ols_val = float(pca_v.values[0]), float(ols_v.values[0])
    status = "OK" if ols_val >= pca_val - 0.02 else "WARN"
    print(f"[{status}] {ds}: OLS-RRR={ols_val:.4f} vs PCA={pca_val:.4f} "
          f"({'OLS-RRR leads' if ols_val >= pca_val else 'PCA leads — investigate'})")

print("\nIf all checks are OK, proceed to Session S2 (GPU deep baselines).")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Per-dataset anomaly ratio and score distribution inspection
# ══════════════════════════════════════════════════════════════════════════════
#
# Before moving to GPU sessions, visually inspect the score distributions
# and anomaly positions for one representative series from each dataset.
# This catches labelling problems (e.g. labels shifted by 1 timestep,
# or anomaly segments too short to be detectable) before you waste GPU time.

import matplotlib.pyplot as plt
import numpy as np
import os

# Must match the separator used in harness.py (_SEP = "___")
_SCORE_SEP  = "___"
SCORES_DIR  = "results/scores"

def plot_series_inspection(series, scores_pca, title=""):
    """Plot raw test signal, PCA anomaly score, and ground-truth labels."""
    T = len(series.test)
    fig, axes = plt.subplots(3, 1, figsize=(14, 6), sharex=True)
    fig.suptitle(title or series.name, fontsize=11)

    axes[0].plot(series.test[:, 0], linewidth=0.5)
    axes[0].set_ylabel("Channel 0 (normalised)")

    axes[1].plot(scores_pca, linewidth=0.5, color='orange')
    axes[1].set_ylabel("PCA score")

    axes[2].fill_between(range(T), series.labels, alpha=0.7, color='red')
    axes[2].set_ylabel("Anomaly label")
    axes[2].set_xlabel("Timestep")
    axes[2].set_ylim(-0.1, 1.3)

    plt.tight_layout()
    plt.show()

for ds_name, records in legacy_data.items():
    if not records:
        continue
    s = records[0]

    # Load PCA scores from the file written by Cell 3 — no recomputation needed.
    # Key format mirrors harness.py save_scores: method___dataset___series___seedN
    key         = f"PCA{_SCORE_SEP}{ds_name}{_SCORE_SEP}{s.name}{_SCORE_SEP}seed42"
    scores_path = os.path.join(SCORES_DIR, f"{key}__scores.npy")

    if os.path.exists(scores_path):
        pca_scores = np.load(scores_path)
        print(f"{ds_name}/{s.name}: loaded PCA scores from disk")
    else:
        print(f"  [WARNING] {s.name}: PCA .npy not found — recomputing (run Cell 3 first).")
        from src.models.baselines.cpu_baselines import pca_anomaly_score
        pca_scores = pca_anomaly_score(s.train, s.test)

    print(f"  anomaly_ratio={s.labels.mean():.3f} | T_test={len(s.test):,} | V={s.V}")
    plot_series_inspection(s, pca_scores, title=f"{ds_name} — {s.name}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — Save session outputs before the session ends
# ══════════════════════════════════════════════════════════════════════════════
#
# IMPORTANT: Click 'Save Version' in Kaggle before this session ends.
# Name the saved version 'ds-patchmamba-run1'.
# The next session (S2) will mount it at /kaggle/input/ds-patchmamba-run1/
# and Cell 1 will restore the results/ and checkpoints/ folders automatically.

import pandas as pd
import os

RESULTS_CSV = "results/main_results.csv"

if os.path.exists(RESULTS_CSV):
    df = pd.read_csv(RESULTS_CSV)
    print(f"Results CSV: {len(df)} rows | {df['method'].nunique()} methods | {df['dataset'].nunique()} datasets")
    print("\nVUS-PR summary:")
    print(df.groupby(['method', 'dataset'])['vus_pr'].mean().unstack().to_string())
else:
    print("No results yet — run Cells 2–4 first.")

scores_dir = "results/scores"
n_scores = len([f for f in os.listdir(scores_dir) if f.endswith('.npy')]) if os.path.exists(scores_dir) else 0
print(f"\nScore files saved: {n_scores}")
print("\n[REMINDER] Click 'Save Version' → name it 'ds-patchmamba-run1' → Save.")